# 3. Tools

This notebook demonstrates how to create and use tools with the language model.

In [1]:
print("Hello world")

Hello world


In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
response = model.invoke("Hello how are you what is your name?")
response.content[-1]

{'type': 'text',
 'text': "Hello! I'm doing well, thank you for asking. \n\nMy name is Gemini. I am a large language model, built by Google. \n\nHow can I help you today?",
 'extras': {'signature': 'EpAHCo0HARFNMg+9JqzZahkc+A1KDSPCJFr7vFdjUXGpOtPEutIbAeJvPWJZC4nPD0Ic0/VlMaRgHRgagmVQQHBUbOdOJiHozv5iDAqXXBhdIQxuwL6LWtp05y5q2n71sfu38gYSRppN7BTrbqphpvdrEMHIBRvhc/u+mhv7dF9hEuXbWMti41Gdpcgrs26tJWyxO/eYisVo26mYEOJKVMfFT3VrXpvmVIFkRptHQQgmLpEszfsYaEFwQozHvStLRfHzfJpLsJE36trnp4OZc2bjOY2c7axUgKLBLfH493MdypJtDApP/pPAAVP+zDgzUEwO5XMIypBGGQBZj3w2Ci2t+/1XSfnTmVkZxO1H2iD82hSe8/+OuOlYNyf52Ig0Haal8TH34yKaqoSzaEI15y9H4idh/xxmjttXoI1jXcmV4N6Beo29fZ5CFLxnzh7xMIkf0S0uGYWc3Oc+qmK/aotkh7yPxbW+o2L2T8jNp/SAQUUpd7DSAkcbQNL+SYumyWAqDWWjC+d9R5We8fgGTJcfaTfmuec4pmJcSsjgxxIjophJ/iuR8lN/Vb+lHUVnT/FFr/tnQ8oA8tUfKg8NpuomPjB52yC8YhUVcrVzGMgqZgOb9atMEtOAhg0aUXgtNTd4UgTmTyPEin46pLHMn+vGYlTr/ZRNxpiLTJEJHWrh+URzXegqoKkiGRTap9m0gCb9CaxpQJuK8R9zgSVKaqwiW8KjC7Xde2ezdN4p9eyge+LZHsoxXBcoO1OaaBlG0K5WTJb7WsTPGufEMtHCqn9jI164iEBnS

In [8]:
from langchain.tools import tool

@tool 
def get_weather(location:str)->str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is sunny and 25 degrees Celsius."

In [12]:
model_with_tools = model.bind_tools([get_weather])

In [13]:
response = model_with_tools.invoke("what is the weather in new york city?"
                                    )

print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"args : {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "new york city"}'}, '__gemini_function_call_thought_signatures__': {'hi3e4zov': 'EtgCCtUCARFNMg9hhrENps7hHGZsu7bbzHpNdPj+AqS8bnmtLrYQW9BLozI20aCW96E71Hfwy/VDA0cSbDXb8txzGWkUXZno/XiK1J1oYvbvUi1LUpGYY2jI4COAAJgaZBaIXRVOCG9PszBYeEpkGLvAMAaUAzSB31NkPVdr8gZUEPG/ql3azdcSouIEkVg0lsoEFD5neqsGViF4+v+hGluIO6iSELi3fy/uwsjJTOyz2TaAhW/7nfXkmgTbx9m6+jxOlzwnqgvZPI9sljcDmsusHK0jUaqQxcwX4/Jf7V58d7h+mzLhEscpojR+D+Or16PQr/cSZG9x9mc0oYB0UOhru2Ut23lCq9isJV3mJRSp4vfnUMz0KsjnsQYMsFlltbNUA1iaDA839x4HCQFiCC8OaRXQs0egYQS7b+b9XM/x7chp2nyUwdVZzdqN3TBs+/eHoceY2bUy5C0='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f898d-baeb-78d1-a631-62bb780c042d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'new york city'}, 'id': 'hi3e4zov', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens'

## tool execution loops

In [14]:
messages = [{
    "role": "user",
    "content": "what is the weather in new york city?"
}]

ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

for tool_call in ai_message.tool_calls:
    tool_res = get_weather.invoke(tool_call)
    messages.append(tool_res)

final_response = model_with_tools.invoke(messages)
print(final_response)

content=[{'type': 'text', 'text': 'The current weather in New York City is sunny and 25 degrees Celsius.', 'extras': {'signature': 'EjQKMgERTTIP/sqOToUIC0lmSDa0guMZdzppWZxNp0ejlDFuz9fhZHKMMEN8WuIy79b0E9DQ'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f8990-9189-77a0-ad9a-7ab2b9e1be9c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 187, 'output_tokens': 16, 'total_tokens': 203, 'input_token_details': {'cache_read': 0}}
